# 07 — Structured Outputs in Agents — Practical Examples

**Covers**: Typed agent decision loop, structured planning, task routing, state management, complete research agent

In [ ]:
import os, json
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Optional
from dotenv import load_dotenv
from rich import print as rprint

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'
print('✅ Setup complete')

## 1. The Agent Decision Schema — Typed Decisions

In [ ]:
# Define the typed interface for agent decisions
class AgentAction(BaseModel):
    thought: str = Field(description="Internal reasoning about what to do next")
    action: Literal["search", "calculate", "answer"]
    action_input: str = Field(description="Query for search/calculate, or final answer text")
    confidence: float = Field(ge=0.0, le=1.0)
    is_final: bool = False

# See what a single step looks like
test_query = "What is the population of Tokyo?"
r = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a research assistant. Decide your next action."},
        {"role": "user",   "content": test_query}
    ],
    response_format=AgentAction
)
step = r.choices[0].message.parsed
print(f"Thought:    {step.thought}")
print(f"Action:     {step.action}")
print(f"Input:      {step.action_input!r}")
print(f"Confidence: {step.confidence:.0%}")
print(f"Is Final:   {step.is_final}")

## 2. Full Structured Agent Loop

In [ ]:
# Mock tools
def search_tool(query: str) -> str:
    db = {
        "tokyo population":   "Tokyo: ~14 million city, ~37 million metro area (2024).",
        "python popularity":  "Python is #1 or #2 most popular language globally (2024 TIOBE index).",
        "openai revenue":     "OpenAI estimated 2024 revenue: $3.4 billion (reported by Bloomberg).",
        "tesla stock":        "Tesla (TSLA) closed at $248.50, +2.3% today."
    }
    for k, v in db.items():
        if any(word in query.lower() for word in k.split()):
            return v
    return f"Search results for '{query}': Relevant information found in multiple sources."

def calculate_tool(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"Calculation error: {e}"

TOOL_MAP = {"search": search_tool, "calculate": calculate_tool}

def structured_agent(question: str, max_steps: int = 5) -> str:
    """Agent loop where every decision is a typed Pydantic object."""
    messages = [
        {"role": "system", "content": """
You are a research assistant with search and calculate tools.
At each step, choose the best action. When you have enough info, set action='answer' with the complete answer.
        """},
        {"role": "user", "content": question}
    ]
    print(f"\n{'═'*60}")
    print(f"  Question: {question}")
    print(f"{'═'*60}")
    
    for i in range(max_steps):
        r = client.beta.chat.completions.parse(
            model=MODEL, messages=messages, response_format=AgentAction
        )
        step = r.choices[0].message.parsed
        
        print(f"\n[Step {i+1}] 💭 {step.thought[:80]}")
        print(f"         🔧 {step.action}({step.action_input[:60]!r})") 
        
        if step.action == "answer" or step.is_final:
            print(f"\n✅ Final Answer:\n{step.action_input}")
            return step.action_input
        
        # Execute tool
        tool_result = TOOL_MAP[step.action](step.action_input)
        print(f"         📦 Result: {tool_result[:100]}")
        
        messages.append({"role": "assistant", "content": json.dumps(step.model_dump())})
        messages.append({"role": "user", "content": f"Tool result: {tool_result}\nContinue or give final answer."})
    
    return "Max steps reached without final answer"

answer = structured_agent("What is the population of Tokyo, and how does it compare to New York (8.3M)?")
print(f"\n{'─'*60}")

## 3. Multi-Step Planner — Generate Plan as Structured Output

In [ ]:
class PlanStep(BaseModel):
    step_num: int
    description: str
    tool: Literal["search", "calculate", "write", "none"]
    input_hint: str = Field(description="What to pass to the tool")
    depends_on: list[int] = Field(default_factory=list)

class ExecutionPlan(BaseModel):
    goal: str
    steps: list[PlanStep]
    total_steps: int
    parallelizable: bool

def plan_task(task: str) -> ExecutionPlan:
    r = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a task planner. Break complex tasks into clear steps."},
            {"role": "user",   "content": f"Create a detailed execution plan for: {task}"}
        ],
        response_format=ExecutionPlan
    )
    return r.choices[0].message.parsed

plan = plan_task("Research the top 3 Python web frameworks and write a comparison")
print(f"Goal: {plan.goal}")
print(f"Steps: {plan.total_steps} | Parallelizable: {plan.parallelizable}")
print()
for s in plan.steps:
    deps = f" (after steps {s.depends_on})" if s.depends_on else ""
    print(f"  Step {s.step_num}: [{s.tool}] {s.description}{deps}")
    print(f"           Input: {s.input_hint[:60]}")

## 4. Task Router — Structured Classification for Agent Routing

In [ ]:
class TaskRoute(BaseModel):
    task_type: Literal["research", "coding", "data_extraction", "math", "creative", "qa"]
    complexity: Literal["simple", "moderate", "complex"]
    needs_tools: bool
    best_agent: Literal["researcher", "coder", "extractor", "calculator", "general"]
    estimated_steps: int = Field(ge=1, le=20)
    priority: Literal["low", "medium", "high"]

def route_request(request: str) -> TaskRoute:
    r = client.beta.chat.completions.parse(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Classify this user request for routing to the appropriate agent."},
            {"role": "user",   "content": request}
        ],
        response_format=TaskRoute
    )
    return r.choices[0].message.parsed

test_requests = [
    "What is 15% of $2,450?",
    "Write a FastAPI endpoint that accepts POST with JSON body and saves to SQLite",
    "Extract all dates mentioned in this document: [contract text here]",
    "Who won the 2024 US presidential election and what were the key issues?"
]

print(f"{'Request':<52} {'Type':<12} {'Agent':<12} {'Steps':<7} {'Priority'}")
print("─" * 100)
for req in test_requests:
    route = route_request(req)
    print(f"{req[:50]:<52} {route.task_type:<12} {route.best_agent:<12} {route.estimated_steps:<7} {route.priority}")

## 5. Serializable Agent State — Persist Between Sessions

In [ ]:
from datetime import datetime
from pydantic import BaseModel, Field
from typing import Optional

class MemoryEntry(BaseModel):
    key: str
    value: str
    confidence: float = Field(ge=0.0, le=1.0)
    added_at_step: int

class AgentSessionState(BaseModel):
    session_id: str
    task: str
    status: Literal["running", "paused", "complete", "error"]
    current_step: int = 0
    memory: list[MemoryEntry] = Field(default_factory=list)
    final_answer: Optional[str] = None
    created_at: str = Field(default_factory=lambda: datetime.now().isoformat())
    updated_at: str = Field(default_factory=lambda: datetime.now().isoformat())

    def remember(self, key: str, value: str, confidence: float = 0.9):
        self.memory.append(MemoryEntry(key=key, value=value, confidence=confidence, added_at_step=self.current_step))
        self.updated_at = datetime.now().isoformat()

    def recall(self, key: str) -> Optional[str]:
        for entry in reversed(self.memory):
            if entry.key == key:
                return entry.value
        return None

    def context_for_llm(self) -> str:
        if not self.memory:
            return "No findings yet."
        return "\n".join(f"- {m.key}: {m.value}" for m in self.memory[-10:])

# Demo: simulate a stateful agent
import uuid
state = AgentSessionState(
    session_id=str(uuid.uuid4())[:8],
    task="Research Python frameworks",
    status="running"
)

state.current_step = 1
state.remember("django_users",   "12M+ websites use Django",   confidence=0.9)
state.current_step = 2
state.remember("fastapi_speed",  "FastAPI is 2-3x faster than Flask", confidence=0.85)
state.current_step = 3
state.remember("flask_maturity", "Flask released in 2010, most mature ecosystem", confidence=0.95)

# Show state
print(f"Session: {state.session_id} | Task: {state.task}")
print(f"Status: {state.status} | Steps: {state.current_step}")
print(f"\nMemory Entries: {len(state.memory)}")
print(state.context_for_llm())

# Serialize to JSON (for persistence to DB or file)
state_json = state.model_dump_json(indent=2)
print(f"\nSerialized state: {len(state_json)} chars")

# Deserialize back
recovered = AgentSessionState.model_validate_json(state_json)
print(f"Recovered recall 'fastapi_speed': {recovered.recall('fastapi_speed')}")

## 6. Complete Research Agent — All Patterns Combined

In [ ]:
class ResearchAction(BaseModel):
    thought: str
    action: Literal["search", "answer"]
    search_query: Optional[str] = None
    key_finding: Optional[str] = None   # What this step revealed
    final_report: Optional[str] = None  # For 'answer' action

def full_research_agent(topic: str, max_steps: int = 5) -> str:
    """Production-style research agent with typed decisions and state."""
    session = AgentSessionState(
        session_id=str(uuid.uuid4())[:8],
        task=topic,
        status="running"
    )
    
    messages = [
        {"role": "system", "content": f"""
Research agent with search tool. Topic: {topic}
Search 2-3 times to gather facts. Then write a complete report with action='answer'.
        """},
        {"role": "user", "content": f"Research comprehensively: {topic}"}
    ]
    
    print(f"\n🔬 Research Agent | Session: {session.session_id}")
    print(f"   Topic: {topic}\n")
    
    for step_num in range(max_steps):
        session.current_step = step_num + 1
        
        # Include agent memory in context
        if session.memory:
            messages[-1]["content"] = (
                f"Continue research on: {topic}\n"
                f"What you know so far:\n{session.context_for_llm()}"
            )
        
        r = client.beta.chat.completions.parse(
            model=MODEL, messages=messages, response_format=ResearchAction
        )
        action = r.choices[0].message.parsed
        
        print(f"  [Step {step_num+1}] 💭 {action.thought[:75]}")
        
        if action.action == "answer" and action.final_report:
            session.status = "complete"
            session.final_answer = action.final_report
            print(f"\n{'─'*60}")
            print(f"📄 Final Report:\n")
            print(action.final_report)
            print(f"{'─'*60}")
            print(f"\n✅ Done | Steps: {session.current_step} | Memory items: {len(session.memory)}")
            return action.final_report
        
        # Execute search
        if action.search_query:
            print(f"           🔍 Search: {action.search_query!r}")
            result = search_tool(action.search_query)
            print(f"           📦 {result[:80]}")
            
            if action.key_finding:
                session.remember(action.search_query, action.key_finding, confidence=0.8)
            
            messages.append({"role": "assistant", "content": json.dumps(action.model_dump())})
            messages.append({"role": "user",      "content": f"Search result: {result}\nContinue or write final report."})
    
    session.status = "error"
    return "Max steps reached"

full_research_agent("The state of Python web frameworks in 2024", max_steps=5)

## 📌 Summary — Structured Outputs in Agents

| Pattern | Purpose | Key Schema |
|---|---|---|
| Agent Decision | Typed decisions at each step | `AgentAction(thought, action, action_input)` |
| Planner | Full task decomposition before execution | `ExecutionPlan(goal, steps[])` |
| Router | Classify + route to specialized agents | `TaskRoute(task_type, best_agent)` |
| Observation | Structured tool results | `ToolObservation(success, result_summary)` |
| State | Persistent, serializable agent memory | `AgentSessionState(memory[], status)` |

**Structured outputs = typed API between agent reasoning and code execution.**